# Unit 6 exercises: grip and the speed limit

A tyre makes sideways force by slipping, and only up to a ceiling.
These exercises take the dynamic bicycle apart by hand, measure the ceiling on a skidpad, turn it into a speed limit along a whole lap, and then ask the questions a racer asks: where exactly did the car spin, and is the margin worth what it costs?
Every number here is a number about racekit's models (`DynamicCar` and `carla_like`), not the truth.

Run the setup cell first.
Each exercise names the variables your answer must define; fill in every `None` and `"TODO"`.
Functions you write in one exercise may be reused in later ones, so run the exercises in order.
Exercises 5, 6 and 8 race; a race that ends in spin-outs runs to its time cap (about 12 s for a 1500 s cap), and Exercise 6 races ten times and takes about a minute.

In [ ]:
import math
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
from racekit import write_variant
from racekit.cars import DynamicCar, KinematicCar, carla_like
from racekit.measure import noise_aware_better, repeat_races
from racekit.runner import run_race
from racekit.tracks import load_track

AGENTS = Path("../../projects/racekit/agents")
UNIT_06 = AGENTS / "unit-06.py"
workdir = Path(mkdtemp(prefix="roar-unit-06-ex-"))

DT = 0.05  # s: one control tick
G = 9.81  # m/s^2
C_DRAG, A_THROTTLE, A_BRAKE = 4.0e-4, 6.0, 8.0  # the longitudinal map (KinematicCar and DynamicCar() share it)
CLEAN_OFFSET = 2.0  # m: the clean rule's limit on max |cross-track error|


def spins(result):
    # How many of the race's respawns were spin-outs.
    return sum(cause == "spin" for cause in result.respawn_causes)


def max_offset(result):
    # The largest |cross-track error| over the driving rows (held == 0).
    telemetry = result.telemetry
    return float(np.max(np.abs(telemetry["offset"][telemetry["held"] == 0])))


def clean(result):
    # The book's clean rule from Unit 6 on: finished, 0 collisions, max |offset| at most 2 m.
    return result.finished and result.collisions == 0 and max_offset(result) <= CLEAN_OFFSET

## Exercise 1 - Slip angles and tyre forces by hand

Use `DynamicCar()`'s parameters: m = 1845 kg, I_z = 2900 kg m², l_f = 1.40 m, l_r = 1.475 m (L = 2.875 m), μ = 1, and the tyre shape B = 10, C = 1.6.
The static axle loads are F_zf = m g l_r / L and F_zr = m g l_f / L.

Write:

- `slip_angles(vx, vy, r, delta, lf=1.40, lr=1.475)`: returns (α_f, α_r) with α_f = atan2(v_y + l_f r, v_x) − δ and α_r = atan2(v_y − l_r r, v_x);
- `tyre_force(alpha, mu, fz, b=10.0, c=1.6)`: returns F_y = −μ F_z sin(c · atan(b α)) (write atan(u) as `math.atan2(u, 1.0)`).

Then, for a car with v_x = 15 m/s, v_y = −0.3 m/s, r = 0.5 rad/s and a front wheel angle δ = 0.12 rad:

1. compute both slip angles and both lateral forces;
2. compute the lateral acceleration the two forces give, (F_yf cos δ + F_yr) / m (ignore the drive forces), and the yaw acceleration from the yaw equation, (l_f F_yf cos δ − l_r F_yr) / I_z;
3. find the slip angle at which the tyre's force peaks, analytically (from C · atan(B α) = π/2), and numerically: the α in `np.linspace(0.0, 0.5, 50001)` with the largest |F_y|;
4. compute the front axle's cornering stiffness, the slope −dF_y/dα at α = 0, analytically (B · C · μ F_zf) and by the central difference (F(−h) − F(h)) / (2h) with h = 1e-6.

Define:

- `slip_angles`, `tyre_force`: the functions;
- `fz_front`, `fz_rear`: the static loads (N);
- `alpha_f`, `alpha_r`: the slip angles (rad);
- `force_f`, `force_r`: the lateral forces (N, left positive);
- `lateral_accel` (m/s²) and `yaw_accel` (rad/s²);
- `alpha_peak`, `alpha_peak_numeric`: the peak slip angle both ways (rad, positive);
- `stiffness_front`, `stiffness_front_numeric`: the front cornering stiffness both ways (N/rad);
- `past_peak`: a dict `{"front": ..., "rear": ...}`, `True` where that axle's |α| is beyond `alpha_peak`;
- `turning_in`: one sentence: is this car's yaw rate growing or shrinking at this instant, and which axle wins?

In [ ]:
def slip_angles(vx, vy, r, delta, lf=1.40, lr=1.475):
    return None  # TODO: (alpha_f, alpha_r)


def tyre_force(alpha, mu, fz, b=10.0, c=1.6):
    return None  # TODO


fz_front, fz_rear = None, None  # TODO
alpha_f, alpha_r = None, None  # TODO
force_f, force_r = None, None  # TODO
lateral_accel = None  # TODO
yaw_accel = None  # TODO
alpha_peak = None  # TODO
alpha_peak_numeric = None  # TODO
stiffness_front = None  # TODO
stiffness_front_numeric = None  # TODO
past_peak = {}  # TODO
turning_in = "TODO"

## Exercise 2 - Where the two models part

The lesson showed `DynamicCar` agreeing with Unit 3's kinematic bicycle at low speed with a gentle steer.
Push the steer harder and look for where they part.

For each speed v in 2.5, 5, 10, 15, 20 and 25 m/s:

- make a `DynamicCar()` and a `KinematicCar(lag_ticks=1)`, and `reset` both to (0, 0, 0) at speed v;
- for 200 ticks, call `advance(hold, -0.05, 0.0, 0.0, 0.0, 0.05, 10)` on both, with the fixed throttle `hold = C_DRAG * v**2 / A_THROTTLE`;
- after every tick on the dynamic car, note whether `spun_out()` is `True`.

After the 200 ticks, the kinematic car's yaw rate is r_k = v_k · tan(0.05 · 70°) / 2.875, with v_k its speed `kinematic.v`; the dynamic car's is its attribute `r`, and its speed is its `v`.

Define:

- `spun`: a dict from speed to `True` when the dynamic car spun out after any of the 200 ticks;
- `yaw_rate_ratio`: a dict from speed to r_dynamic / r_kinematic, or `None` for a speed at which the car spun;
- `curvature_ratio`: a dict from speed to (r_dynamic / v_dynamic) / (r_kinematic / v_kinematic), or `None` where the car spun;
- `speed_ratio`: a dict from speed to v_dynamic / v, or `None` where the car spun;
- `agree_to_1_percent`: the largest speed whose `yaw_rate_ratio` is within 1% of 1;
- `demanded_at_25`: the lateral acceleration the kinematic car's circle asks for at 25 m/s, 25² · tan(0.05 · 70°) / 2.875 (m/s²);
- `where_they_part`: one or two sentences: the dynamic car keeps the kinematic car's path curvature but not its yaw rate at 10–20 m/s; why, and what happens at 25 m/s?

In [ ]:
spun = {}  # TODO
yaw_rate_ratio = {}  # TODO
curvature_ratio = {}  # TODO
speed_ratio = {}  # TODO
agree_to_1_percent = None  # TODO
demanded_at_25 = None  # TODO
where_they_part = "TODO"

## Exercise 3 - The skidpad on a wet track

Repeat the lesson's skidpad at lower grip.
The procedure, exactly:

- `car = DynamicCar(mu=mu)`, `car.reset(0.0, 0.0, 0.0, 5.0)`;
- for tick n = 0, 1, 2, … (at most 4000 ticks): the speed goal is 5.0 + 0.2 · n · 0.05; the throttle is `min(max(0.5 * (goal - car.v) + C_DRAG * car.v**2 / A_THROTTLE, 0.0), 1.0)`; keep `vx, vy = car.vx, car.vy`; call `car.advance(throttle, -0.1, 0.0, 0.0, 0.0, 0.05, 10)`;
- after the call, a_lat = (car.vy − vy) / 0.05 + car.vx · car.r;
- stop after the first tick on which `car.spun_out()` is `True`.

Run it for μ = 0.5, 0.7 and 1.0.
At low speed the car follows the kinematic circle of curvature κ₀ = tan(0.1 · 70°) / 2.875, so the friction circle predicts the spin near v = √(μg / κ₀).

Define:

- `peak_lat`: a dict from μ to the largest |a_lat| of the run (m/s²);
- `peak_fraction`: a dict from μ to `peak_lat[mu] / (mu * G)`;
- `spin_speed`: a dict from μ to the car's speed `car.v` after the spin tick (m/s), or `None` if it did not spin within 4000 ticks;
- `predicted_speed`: a dict from μ to √(μg / κ₀) (m/s);
- `fig`, `ax`: one plot of |a_lat| against the speed for the three runs, with a horizontal line at μg for each, axis labels, a legend and a title;
- `scales_with_mu`: `True` when every `peak_fraction` is between 0.95 and 1.0;
- `wet_track`: one sentence: what does halving μ do to the fastest speed round a given corner, and why the square root?

In [ ]:
peak_lat = {}  # TODO
peak_fraction = {}  # TODO
spin_speed = {}  # TODO
predicted_speed = {}  # TODO
fig, ax = None, None  # TODO
scales_with_mu = None  # TODO
wet_track = "TODO"

## Exercise 4 - The speed cap along a lap

Build the agent's speed cap for a whole track, without racing.

Write:

- `menger_curvature(xs, ys)`: Unit 2's signed curvature at every waypoint from the cyclic triple (i − 1, i, i + 1), left positive, as a NumPy array;
- `curvature_ahead(kappa, horizon=25)`: at each waypoint i, the largest |κ| over waypoints i, i + 1, …, i + horizon − 1 (cyclic), as a NumPy array;
- `speed_cap(kappa_ahead, a_lat, target=20.0)`: at each waypoint, min(target, √(a_lat / κ_ahead)), or `target` where κ_ahead ≤ 1e-9, as a NumPy array.

On `monza-like` (`load_track("monza-like")`, coordinates `track.x` and `track.y`) with `A_LAT = 6.0`:

1. compute the cap at every waypoint;
2. compute the **cap lap time**: the time to drive one lap if the car's speed on segment i (from waypoint i to waypoint i + 1, the last segment closing the lap) were exactly the cap at waypoint i, Σ length_i / cap_i;
3. compare it with the flat lap time at 20 m/s, the lap length / 20.

On the `oval`: the cap is below 20 m/s somewhere on the lap only when A_LAT < 20² · max |κ|.
Compute that threshold.

Define:

- `menger_curvature`, `curvature_ahead`, `speed_cap`: the functions;
- `monza_cap`: the cap array on `monza-like`;
- `capped_fraction`: the fraction of `monza-like`'s waypoints at which the cap is below 20 m/s;
- `slowest_cap`, `slowest_index`: the smallest cap (m/s) and the first waypoint index where it occurs;
- `cap_lap_time`, `flat_lap_time`: the two lap times (s);
- `oval_binding_a_lat`: the oval's threshold (m/s²);
- `why_oval`: one sentence: what does `oval_binding_a_lat` say about racing the cap on the oval at any A_LAT from 5 to 10?

In [ ]:
def menger_curvature(xs, ys):
    return None  # TODO


def curvature_ahead(kappa, horizon=25):
    return None  # TODO


def speed_cap(kappa_ahead, a_lat, target=20.0):
    return None  # TODO


monza_cap = None  # TODO
capped_fraction = None  # TODO
slowest_cap, slowest_index = None, None  # TODO
cap_lap_time = None  # TODO
flat_lap_time = None  # TODO
oval_binding_a_lat = None  # TODO
why_oval = "TODO"

## Exercise 5 - Reading a spin-out from the telemetry

Make a variant of `agents/unit-06.py` with `SPEED_MODE = "flat"` (Unit 5's agent; use `write_variant` into `workdir`) and race it on `"test-circuit"` with `car=carla_like(noise=False)`, `seed=0` and `max_seconds=1500.0`.

Find the first spin-out and what the car was asking of its tyres just before it:

1. the first respawn's time is `result.respawn_times[0]` (check with `result.respawn_causes[0]` that it was a spin);
2. take the **last telemetry row with t below that time and `held == 0`**; read its x, y and v;
3. find the waypoint of `test-circuit` nearest to that (x, y) (the smallest straight-line distance over all waypoints);
4. compute the curvature ahead there with your Exercise 4 functions (horizon 25, from that waypoint);
5. compute the lateral acceleration a steady turn at that speed and curvature would demand, v² · κ_ahead, and the speed the lesson's carla-like clean edge, A_LAT = 7, would have allowed there, √(7 / κ_ahead).

Define:

- `flat_result`: the race result;
- `spin_count`: its number of spin-outs;
- `first_spin_time`: the first respawn's time (s);
- `spin_x`, `spin_y`, `spin_v`: the row's position (m) and speed (m/s);
- `spin_waypoint`: the nearest waypoint's index;
- `kappa_there`: the curvature ahead there (1/m);
- `demanded_a_lat`: v² · κ_ahead (m/s²);
- `allowed_speed`: √(7 / κ_ahead) (m/s);
- `over_by`: `spin_v / allowed_speed`;
- `diagnosis`: one or two sentences: was the spin a speed problem or a steering problem, and what number in your answer shows it?

In [ ]:
flat_result = None  # TODO
spin_count = None  # TODO
first_spin_time = None  # TODO
spin_x, spin_y, spin_v = None, None, None  # TODO
spin_waypoint = None  # TODO
kappa_there = None  # TODO
demanded_a_lat = None  # TODO
allowed_speed = None  # TODO
over_by = None  # TODO
diagnosis = "TODO"

## Exercise 6 - Is the margin worth it?

On `carla_like(noise=False)` the clean edge on `test-circuit` is A_LAT = 7, and the reference ships A_LAT = 6 (the margin rule).
Test the rule on the **noisy** car, with Unit 1's measurement rule.

Race `agents/unit-06.py` (A_LAT 6) and a variant of it with `A_LAT = 7.0` (made with `write_variant`), each with `repeat_races` on `"test-circuit"`, `n=5`, seeds 0–4, `car_factory` returning `carla_like(noise=True)`, and `max_seconds=600.0` (a clean race finishes in about 420 s; a race still running at 600 s is not clean).

Define:

- `clean_6`, `clean_7`: lists of the five per-seed clean flags (the setup cell's `clean`), seed order;
- `spins_6`, `spins_7`: lists of the five per-seed spin-out counts, seed order;
- `median_6`, `spread_6`: the median and spread (max − min) of A_LAT 6's race times (s);
- `clean_times_7`: the race times of A_LAT 7's **clean** races only, in seed order;
- `ship`: the A_LAT to ship by this rule: a setting is eligible only if it is clean on all five seeds; among the eligible settings, ship the one with the lower median time (6.0 or 7.0);
- `edge_on_noise`: one or two sentences: what did one deterministic clean race at A_LAT 7 fail to tell you, and what does that say about choosing a setting at the edge?

In [ ]:
clean_6, clean_7 = [], []  # TODO
spins_6, spins_7 = [], []  # TODO
median_6, spread_6 = None, None  # TODO
clean_times_7 = []  # TODO
ship = None  # TODO
edge_on_noise = "TODO"

## Exercise 7 - Challenge: braking in the corner

The friction circle says braking uses grip the corner needs.
On a real car, braking in a corner can make the car spin.
Does it on `DynamicCar()`?

The procedure, for each brake level b in 0.2, 0.4, 0.6, 0.8 and 1.0:

- `car = DynamicCar()`, `car.reset(0.0, 0.0, 0.0, 14.0)`;
- **hold** for 400 ticks: the throttle is `min(max(0.5 * (14.0 - car.v) + C_DRAG * car.v**2 / A_THROTTLE, 0.0), 1.0)`, the steer −0.1, the brake 0;
- then **brake** for 80 ticks: throttle 0, steer −0.1, brake b;
- every tick call `car.advance(throttle, -0.1, brake, 0.0, 0.0, 0.05, 10)`, keeping `vx, vy = car.vx, car.vy` before the call, and compute after it a_long = (car.vx − vx) / 0.05 − car.vy · car.r and a_lat = (car.vy − vy) / 0.05 + car.vx · car.r;
- stop early if `car.spun_out()` is `True` after a braking tick (`spun_out` counts only above 5 m/s: as the car comes to rest at the heavier brake levels its sideslip angle can grow large without that being a spin).

Define:

- `lat_before`: a_lat on the last hold tick (the same for every b; m/s²);
- `spun_by_brake`: a dict from b to `True` when the car spun out during the braking ticks;
- `peak_total`: a dict from b to the largest √(a_long² + a_lat²) over the braking ticks (m/s²);
- `peak_fraction`: a dict from b to `peak_total[b] / G`;
- `room_for_lat`: a dict from b to √(G² − (A_BRAKE·b)²), the lateral acceleration the circle leaves at a braking deceleration of A_BRAKE·b (m/s²);
- `smallest_spin_brake`: the smallest b that spun, or `None`;
- `room_short_from`: the smallest b whose `room_for_lat[b]` is below `abs(lat_before)` (from there on, the circle leaves less room than the corner was using), or `None`;
- `why`: one or two sentences: what does this car do instead of spinning, and which of the model's labelled limits (lesson section 2) is the reason a real car behaves differently?

In [ ]:
lat_before = None  # TODO
spun_by_brake = {}  # TODO
peak_total = {}  # TODO
peak_fraction = {}  # TODO
room_for_lat = {}  # TODO
smallest_spin_brake = None  # TODO
room_short_from = None  # TODO
why = "TODO"

## Exercise 8 - Challenge: how far ahead should the cap look?

`agents/unit-06.py` takes the curvature ahead over `CURVATURE_HORIZON = 25` waypoints (about 50 m).
What if it looked only 10 waypoints (about 20 m) ahead?
You have a budget of **three races**.

Race variants of `agents/unit-06.py` with `CURVATURE_HORIZON = 10` and `A_LAT` = 6.0, 7.0 and 8.0 on `"test-circuit"`, `car=carla_like(noise=False)`, `seed=0`, `max_seconds=1500.0`.
Compare with the lesson's horizon-25 row on the same car (probe record (plan roar-006)): A_LAT 6 clean in 424.30 s, A_LAT 7 clean in 418.40 s with a max |offset| of 1.13 m, and A_LAT 8 spinning 19 times.

Define:

- `horizon_10`: a dict from A_LAT to a tuple (clean, race time in s, spin-outs, max |offset| in m);
- `edge_10`: the largest of the three A_LAT values whose race is clean (`None` if none is);
- `same_edge`: `True` when `edge_10` equals the horizon-25 edge, 7.0;
- `faster_at_7`: `True` when the horizon-10 race at A_LAT 7 is clean and faster than 418.40 s;
- `offset_cost_at_7`: the horizon-10 max |offset| at A_LAT 7 minus 1.13 (m);
- `horizon_verdict`: one or two sentences: what does the shorter horizon buy and what does it cost, why (think about when the car starts to brake), and what can three single-seed races not tell you?

In [ ]:
horizon_10 = {}  # TODO
edge_10 = None  # TODO
same_edge = None  # TODO
faster_at_7 = None  # TODO
offset_cost_at_7 = None  # TODO
horizon_verdict = "TODO"